# Bedding × Dinomaly — results & comparison tutorial

Post-hoc analysis of the ALL6 high-res (672×672, 20-epoch) pilot. No GPU required — everything here reads cached JSON / PNG artefacts produced by `eval_bedding_all6.py` and the comparison hub scripts.

Covers:
1. Headline metric table (Dinomaly vs EfficientAD baseline).
2. Per-class pixel AUROC bar chart.
3. ROC curves (overall + per-class) + confusion matrices.
4. Cross-method comparison story: where ALL6 wins, where it still trails EAD.
5. Outstanding investigations checklist (matches `comparisons/README.md`).

In [ ]:
%matplotlib inline
import json, sys
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('.').resolve()))
from utils import resolve_default_config, plot_per_class_auroc_bar, load_headline_report

config = resolve_default_config()
eval_dir = config['eval_dir']
print('Eval dir:', eval_dir)
print('Contents:', sorted(p.name for p in eval_dir.glob('*.json'))[:6])

## 1 · Headline metrics

Loaded from `eval_val/report.json`. These are the val-set numbers the pilot produced.
Note: the cross-method comparison numbers (vs EAD, vs RGB pilots) live in `comparisons/headline_matrix.md` — we surface the key ones here.

In [ ]:
report = load_headline_report(eval_dir / 'report.json')
# Headline table
print(f"{'Metric':<35} {'Value':>12}")
print('-' * 50)
for k in ('pixel_auroc', 'image_auroc', 'image_auroc_p99_9', 'dice_optimal_f1_raw',
          'dice_threshold_raw', 'mean_per_class_auroc'):
    if k in report:
        v = report[k]
        print(f'{k:<35} {v:>12.4f}' if isinstance(v, (int, float)) else f'{k:<35} {v!s:>12}')

# Vs EAD baseline (numbers from comparisons/headline_matrix.md — these are pinned)
ead_baseline = {
    'pixel_auroc': 0.910,
    'image_auroc_mask_labels': 0.846,
    'image_auroc_filename_labels': 0.9396,
    'dice_optimal_f1': 0.679,
    'mean_per_class_auroc': 0.914,
}
print()
print('EAD baseline (reproduction):')
for k, v in ead_baseline.items():
    print(f'  {k:<33} {v:>12.4f}')

## 2 · Per-class pixel AUROC

23 classes, computed using EAD's methodology (per-frame normalize + background drawn from frames where the class is present). Sorted ascending so the weak classes are highlighted at the bottom.

In [ ]:
per_class_json = list(eval_dir.glob('per_class_auroc_*.json'))
if per_class_json:
    fig = plot_per_class_auroc_bar(per_class_json[0],
                                   title='Per-class pixel AUROC — ALL6 high-res (672, 20 ep)')
    fig.savefig(Path('outputs') / 'per_class_auroc_bar.png', dpi=120, bbox_inches='tight')
else:
    print('No per_class_auroc_*.json found. Run recompute_per_class_auroc.py first.')

## 3 · ROC curves + confusion matrices

Pre-rendered PNGs from `eval_val/` and `comparisons/confusion_matrices/`. We just display them inline.

In [ ]:
CM_DIR = Path('/home/dev/anish/cuvis-ai-cookbook/examples/bedding_dinomaly/comparisons/confusion_matrices')
tiles = [
    ('Pixel ROC (overall)', eval_dir / 'roc_pixel.png'),
    ('Per-class ROC', eval_dir / 'roc_per_class.png'),
    ('Pixel CM @ best-F1 thr', CM_DIR / 'all6_highres_20ep_pixel_cm.png'),
    ('Image CM (filename labels, EAD-style)', CM_DIR / 'all6_highres_20ep_image_best_F1_filename.png'),
]
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
for ax, (title, path) in zip(axes.flat, tiles):
    if path.is_file():
        ax.imshow(mpimg.imread(str(path)))
        ax.set_title(title)
    else:
        ax.text(0.5, 0.5, f'(missing)\n{path}', ha='center', va='center')
    ax.axis('off')
fig.tight_layout()
fig.savefig(Path('outputs') / 'results_grid.png', dpi=110, bbox_inches='tight')

## 4 · Cross-method comparison

Big-picture story across the four pilots in `comparisons/headline_matrix.md`. ALL6 high-res beats EAD on every aggregate metric except Dice; the remaining Dice gap is a precision problem (FP pool at the global threshold) not a signal problem.

Where ALL6 high-res wins big over EAD reproduction (per-class pixel AUROC):

- PLA_black_16mm: +0.420
- transparent_plastic: +0.344
- alcohol: +0.123
- water&alcohol-tray: +0.082
- mean across all 23 classes: +0.059

Where ALL6 still trails EAD (these are the open work items):

- **PLA_white_2mm**: 0.835 (vs EAD 0.940). White-on-white visible-light failure mode; aspect-preserving pilots (434×1036, 504×1204) target this directly.
- **PLA_blue_2mm**: 0.908 (vs EAD 0.972). Mid-size blue plastics — similar small-object hypothesis.
- **Dice**: 0.615 (vs EAD 0.679). Likely closes via morphological cleanup of the FP pool at the global threshold — no retraining needed.

## 5 · Outstanding investigations

From `comparisons/README.md` — re-stated here so the notebook is a self-contained portal:

- [ ] Re-annotate `frame_10_nok_ok_rdx_rwx` (the one frame whose annotation JSON has zero polygons, so its mask PNG was never generated). Closes the 0.85 → 0.94 mask-vs-filename label gap permanently.
- [ ] Aspect-preserving pilots (434×1036, 504×1204) — currently training; results land in this notebook's input dir once done.
- [ ] CIR (450 / 650 / 1050 nm) + SWIR-only (1050 / 1200 / 1450 nm) ablations — falsifies whether ALL6's lift comes from SWIR alone or from VIS+SWIR fusion.
- [ ] Morphological-opening cleanup on score map — likely closes most of the remaining Dice gap to EAD.
- [ ] Identify the 2 FN frames at the best image-F1 threshold (which classes do the misses contain?).
- [ ] Curate qualitative-story sample images — narrative around the 8 attached frames on the Jira ticket.
- [ ] Upstream generalisation of `cuvis_ai.node.channel_selector.FixedWavelengthSelector` to n-channels (the port-spec is locked to 3 today; our `FixedHyperspectralSelector` exists solely as a plugin-local workaround).

All deferred to a future ticket / PR cycle.